### Load in Protein Data

In [34]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np

# Load files on Evan's Laptop
# expr_orig = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\expression.csv", index_col=0)
# expr=expr_orig.transpose()
# metadata = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\metadata.csv", index_col=0)
# umap = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\umap.csv", index_col=0)

#Load files on Lab computer
expr_orig = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\expression.csv", index_col=0)
expr=expr_orig.transpose()
metadata = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\metadata.csv", index_col=0)
umap = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\umap.csv", index_col=0)

# Create AnnData object
adata = sc.AnnData(X=expr.values)

# Assign metadata
adata.obs = metadata
adata.var_names = expr.columns
adata.obs_names = expr.index

# Add spatial coordinates and UMAP to .obsm
# adata.obsm["spatial"] = metadata[['x_FOV_px', 'y_FOV_px']].values  # adjust if needed
adata.obsm["spatial"] = metadata[['x_FOV_px']].assign(y_FOV_px = -metadata['y_FOV_px']).values
adata.obsm["X_umap"] = umap.values

### Functions:
1. Find Neighbooring Cell Matrix
2. Find Overall Cell Matrix

In [35]:
def create_neighborhood_matrix(adata,pmn_threshold,spatial_method,radius=15):

    import my_functions
    import pandas as pd
    
    # Initialize collection DataFrames
    all_total_counts_R = pd.DataFrame()
    all_total_counts_NR = pd.DataFrame()
    all_total_counts_R_percent = pd.DataFrame()
    all_total_counts_NR_percent = pd.DataFrame()
    
    # mininum number of pmn cells
    pmn_floor=pmn_threshold
    radius=radius # Radius in microns
    
    for i in range(1, 4):
        for j in range(1, 26):
            sample_id = f"c_{i}_{j}_"
            pmn_count = my_functions.pmn_counter(adata, sample_id)["PMN Count"]
            if pmn_count <= pmn_floor:
                continue
    
            # Get results from your function
            results = my_functions.average_cell_counts(adata, i, j, "radius", radius)
            total_counts = results["Total Cell Counts"]
    
            # Extract total cell count value (from the "Total Cell Count" column)
            total_cell_count = total_counts["Total Cell Count"].values[0]
    
            # Drop that column before calculating percentages
            counts_no_total = total_counts.drop(columns=["Total Cell Count"])
    
            # Calculate percentages of each cell type
            percent_row = (counts_no_total / total_cell_count) * 100
    
            # Append to appropriate response group
            response = my_functions.get_sample_info(adata, sample_id)["Response"]
            if response == "R":
                all_total_counts_R = pd.concat([all_total_counts_R, total_counts])
                all_total_counts_R_percent = pd.concat([all_total_counts_R_percent, percent_row])
            elif response == "NR":
                all_total_counts_NR = pd.concat([all_total_counts_NR, total_counts])
                all_total_counts_NR_percent = pd.concat([all_total_counts_NR_percent, percent_row])

            # Replace NaNs with 0 for safety
            all_total_counts_R.fillna(0, inplace=True)
            all_total_counts_NR.fillna(0, inplace=True)
            all_total_counts_R_percent.fillna(0, inplace=True)
            all_total_counts_NR_percent.fillna(0, inplace=True)

    return{
        "Responder Percentile Matrix":all_total_counts_R_percent,
        "Non-Responder Percentile Matrix":all_total_counts_NR_percent
    }

def get_cell_type_percentages(adata,pmn_threshold):

    import my_functions
    import pandas as pd


    #initialize output
    output_percent_R=pd.DataFrame()
    output_percent_NR=pd.DataFrame()
    output_percent=pd.DataFrame()
    pmn_floor=pmn_threshold

    # Iterate through 
    for i in range(1,4):
        for j in range(1,26):

            #Set sample Id in format
            sample_id=f"c_{i}_{j}_"
            pmn_count = my_functions.pmn_counter(adata, sample_id)["PMN Count"]
            # Pmn floor is typically 25
            if pmn_count <= pmn_floor:
                continue
            percentages = my_functions.get_sample_cell_percentages(adata, sample_id)["Percentage Counts"]
            # Append the percentage row to the overall DataFram
            output_percent=pd.concat([output_percent,percentages])
            if my_functions.get_sample_info(adata,sample_id)["Response"]=="R":
                output_percent_R = pd.concat([output_percent_R, percentages])
            elif my_functions.get_sample_info(adata,sample_id)["Response"]=="NR":
                output_percent_NR = pd.concat([output_percent_NR, percentages])

    # Replaces Nans with 0 to pererve the numerical type of the matrix
    output_percent.fillna(0, inplace=True)
    output_percent_R.fillna(0, inplace=True)
    output_percent_NR.fillna(0, inplace=True)
    
    # Reorder Columns Alphabetically
    output_percent = output_percent[sorted(output_percent.columns)]
    output_percent_R = output_percent_R[sorted(output_percent_R.columns)]
    output_percent_NR = output_percent_NR[sorted(output_percent_NR.columns)]
    
    return{
        "All Percentage": output_percent,
        "R Percentage": output_percent_R,
        "NR Percentage": output_percent_NR
    }

### Detla Matrix Creation

In [51]:
import my_functions
import pandas as pd

pmn_threshold=25
radius_um=45

# Call Functions
overall_cell_types=get_cell_type_percentages(adata,pmn_threshold)
neighborhood_matrix=create_neighborhood_matrix(adata,pmn_threshold,"radius",radius_um)

# Assign Variables
overall_cell_types_R=overall_cell_types["R Percentage"]
overall_cell_types_NR=overall_cell_types["NR Percentage"]
neighborhood_matrix_R=neighborhood_matrix["Responder Percentile Matrix"]
neighborhood_matrix_NR=neighborhood_matrix["Non-Responder Percentile Matrix"]

display(overall_cell_types_R)
display(neighborhood_matrix_R)

# Standardize Matrix
neighborhood_matrix_R.index.name = 'Sample ID'
overall_cell_types_R.index.name = 'Sample ID'
neighborhood_matrix_NR.index.name = 'Sample ID'
overall_cell_types_NR.index.name = 'Sample ID'

overall_cell_types_R.index = overall_cell_types_R.index.str.rstrip('_')
overall_cell_types_NR.index = overall_cell_types_NR.index.str.rstrip('_')

# Create a diff matrix
diff_responder=neighborhood_matrix_R-overall_cell_types_R
diff_non_responder=neighborhood_matrix_NR-overall_cell_types_NR

# Calculate the average across rows (column-wise mean)
average_row_r = pd.DataFrame([diff_responder.mean()], index=["Average"])
average_row_nr = pd.DataFrame([diff_non_responder.mean()], index=["Average"])

# Append it to the original DataFrame
diff_responder_with_avg = pd.concat([diff_responder, average_row_r])
diff_non_responder_with_avg = pd.concat([diff_non_responder, average_row_nr])

# Combine responder and non-responder groups
delta_neighborhood_percent_R = diff_responder_with_avg.copy()
delta_neighborhood_percent_NR = diff_non_responder_with_avg.copy()

# Preserve index as 'sample' for identification
delta_neighborhood_percent_R['sample'] = delta_neighborhood_percent_R.index
delta_neighborhood_percent_NR['sample'] = delta_neighborhood_percent_NR.index

delta_neighborhood_percent_R['response'] = 'R'
delta_neighborhood_percent_NR['response'] = 'NR'

delta_full_neighboor_matrix = pd.concat([delta_neighborhood_percent_R, delta_neighborhood_percent_NR])

merged_annot_cluster,B_cells,CD4+T_cells,CD8+T_cells,DCs,Endothelial_cells,Fibroblasts/SMCs,Macrophages,Monocytes,NK_cells,Neutrophils,Plasma_cells,Tregs,Tumor_cells
c_1_1_,0.100376,0.112923,0.188206,0.000000,41.819322,12.321205,0.501882,0.062735,0.000000,0.690088,1.568381,0.025094,42.609787
c_1_11_,3.732609,6.786563,5.632847,4.343400,0.000000,13.776722,20.732949,8.245674,0.033933,4.004072,7.804547,2.782491,22.124194
c_1_12_,2.422270,4.591468,6.543745,2.458424,0.000000,36.912509,7.230658,2.241504,0.036153,1.373825,4.699928,2.530730,28.958785
c_1_13_,0.572435,3.522677,1.144870,2.642008,0.000000,17.965654,6.428886,4.007045,0.044033,1.629238,1.144870,1.232937,59.665346
c_1_16_,1.421009,5.377542,6.269156,2.981332,0.139315,20.507105,7.049317,3.148509,0.000000,0.919476,15.324603,2.563388,34.299248
c_1_20_,0.729614,2.703863,0.557940,2.188841,1.459227,54.334764,3.519313,2.403433,0.257511,3.433476,6.523605,1.030043,20.858369
c_1_21_,1.838509,5.366460,7.055901,5.987578,0.000000,5.913043,20.298137,6.360248,0.198758,12.298137,5.515528,2.161491,27.006211
c_1_24_,1.220044,4.008715,0.784314,4.880174,0.000000,9.281046,5.838780,2.570806,0.087146,6.405229,0.740741,2.396514,61.786492
c_2_4_,3.719152,5.908933,10.914147,4.657629,0.973236,30.100799,12.964894,4.101495,0.417101,1.529371,3.023983,4.136253,17.553007
c_2_5_,5.172414,8.709107,12.820513,9.372237,2.785146,19.407604,13.527851,4.995579,0.198939,2.210433,3.183024,4.907162,12.709991


,Endothelial_cells,Neutrophils,Tumor_cells,Fibroblasts/SMCs,Plasma_cells,Monocytes,B_cells,Macrophages,CD4+T_cells,DCs,CD8+T_cells,Tregs,NK_cells
Sample ID,,,,,,,,,,,,,
c_1_1,37.856258,14.869888,26.115242,13.630731,5.204461,1.177200,0.278810,0.805452,0.061958,0.000000,0.000000,0.000000,0.000000
c_1_11,0.000000,8.663883,28.627349,12.160752,5.767223,10.594990,2.009395,16.858038,4.618998,6.236952,3.262004,1.200418,0.000000
c_1_12,0.000000,5.685131,41.326531,15.670554,5.174927,1.822157,4.008746,6.997085,4.810496,3.644315,8.600583,2.259475,0.000000
c_1_13,0.000000,23.859649,45.175439,12.017544,0.614035,3.070175,0.526316,4.824561,3.684211,4.824561,0.350877,1.052632,0.000000
c_1_16,0.067981,2.039429,19.850442,23.113528,18.014956,4.826649,1.699524,8.429640,7.002039,4.214820,6.798097,3.942896,0.000000
c_1_20,0.726895,24.402908,42.263759,18.068536,3.634476,2.699896,0.103842,4.984424,1.090343,1.609553,0.000000,0.363448,0.051921
c_1_21,0.000000,31.732661,14.060828,4.608693,5.019505,5.965409,1.567301,14.602824,4.266924,9.849139,6.514309,1.543135,0.269272
c_1_24,0.000000,45.458152,34.172947,3.649345,0.178501,1.209837,0.376835,2.221341,1.348671,10.630702,0.138834,0.614835,0.000000
c_2_4,0.667161,2.520385,15.789474,21.719792,3.558191,3.780578,3.335804,12.824314,6.968125,8.080059,13.120830,7.042254,0.593032


### Get P Values

In [52]:
import pandas as pd
from scipy.stats import ttest_ind

input_matrix=delta_full_neighboor_matrix.copy()

# Separate responder and non-responder groups
responder_matrix = input_matrix[input_matrix['response'] == 'R']
non_responder_matrix = input_matrix[input_matrix['response'] == 'NR']

# List of columns to test (excluding 'response' column)
cell_type_columns = [col for col in input_matrix.columns if col not in ['sample', 'response']]

# Store results
results = []

for cell_type in cell_type_columns:
    r_data = responder_matrix[cell_type]
    nr_data = non_responder_matrix[cell_type]
    
    t_stat, p_value = ttest_ind(r_data, nr_data, equal_var=False) 
    
    results.append({
        'cell_type': cell_type,
        't_statistic': t_stat,
        'p_value': p_value
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
display(results_df)


,cell_type,t_statistic,p_value
0,B_cells,0.102249,0.919258
1,CD4+T_cells,-1.263280,0.213510
2,CD8+T_cells,-1.547764,0.129695
3,DCs,-1.267560,0.216274
4,Endothelial_cells,0.719020,0.478506
5,Fibroblasts/SMCs,-1.054821,0.297577
6,Macrophages,-0.678179,0.501609
7,Monocytes,-0.940712,0.352992
8,NK_cells,-0.834245,0.409014
9,Neutrophils,0.441370,0.661422
